
***STATISTICAL SIGNIFICANCE TESTING FOR A/B TEST RESULTS***

After exploring our data with SQL queries, we now need to determine if the
differences we observed between Group A and Group B are statistically significant
or just due to random chance. 

In [ ]:
1 - Importing Libraries & Loading Data
# -----------------------------------------------------------------------
# pandas and numpy are the basics - dataframes and math
# scipy and statsmodels are what we actually use for the statistical tests
# chi2_contingency -> for testing if conversion difference is real
# ttest_ind -> for comparing page views and time spent between the two groups
# proportion_confint -> to calculate confidence intervals

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind
from statsmodels.stats.proportion import proportion_confint
import warnings
warnings.filterwarnings('ignore')

# connecting to snowflake directly
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE AB_TESTING_DB").collect()
session.sql("USE SCHEMA AB_TESTING_SCHEMA").collect()

# pulling the table straight into a pandas dataframe
df = session.table("AB_TEST_RESULTS").to_pandas()

# snowflake returns column names in all caps, so we rename them to be readable
df.columns = ['User ID', 'Group', 'Page Views', 'Time Spent', 'Conversion', 'Device', 'Location']

print("libraries loaded!")
print(f"loaded {len(df)} rows!")
print(df.head())

In [ ]:
# Splitting into Group A and Group B
# -----------------------------------------------------------------------
# Before running any tests, we need to separate the two groups for comparison.
# The dataset already has users randomly assigned to Group A or Group B.
# Our analysis will determine which group performed better.

group_a = df[df['Group'] == 'A']
group_b = df[df['Group'] == 'B']

print(f"Group A users: {len(group_a)}")
print(f"Group B users: {len(group_b)}")


In [ ]:
# 3 - Chi-Square Test
# -----------------------------------------------------------------------
# The main question of any A/B test is: is the difference we're seeing real,
# or could it just be random luck?
# The chi-square test answers exactly that for yes/no outcomes like conversion.
#
# How it works: We build a contingency table (converted vs not converted
# for each group), and chi-square checks if the pattern is too strong
# to be explained by chance alone.
#
# The result we care about is the p-value:
# - p < 0.05 means the difference is statistically significant (95% confident)
# - p < 0.001 means it's extremely significant (99.9% confident - basically impossible by chance)
# - p >= 0.05 means we can't confidently say there's a real difference
#
# This validates what we found in our SQL analysis:
# Group A had 5.4% conversion vs Group B's 14.07% - but is it significant?
# ----------------------------------------------------------------------------
print("\n[1] CHI-SQUARE TEST FOR CONVERSION RATES")
print("-" * 80)

# Separate data by group
group_a = df[df['Group'] == 'A']
group_b = df[df['Group'] == 'B']

# Count conversions for each group
conversions_a = group_a['Conversion'].sum()
conversions_b = group_b['Conversion'].sum()
non_conversions_a = len(group_a) - conversions_a
non_conversions_b = len(group_b) - conversions_b

contingency_table = np.array([
    [conversions_a, non_conversions_a],
    [conversions_b, non_conversions_b]
])

# Run Chi-Square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Contingency Table:")
print(f"                 Converted    Not Converted    Total")
print(f"Group A          {conversions_a:>9}    {non_conversions_a:>13}    {len(group_a):>5}")
print(f"Group B          {conversions_b:>9}    {non_conversions_b:>13}    {len(group_b):>5}")
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Degrees of freedom: {dof}")

# Interpret p-value
# p < 0.001: Highly significant (99.9% confident)
# p < 0.05: Significant (95% confident) - industry standard
# p >= 0.05: Not significant (could be random)

if p_value < 0.001:
    print(f"\n RESULT: Highly statistically significant (p < 0.001)")
    print(f"   The conversion rate difference is NOT due to random chance.")
elif p_value < 0.05:
    print(f"\n RESULT: Statistically significant (p < 0.05)")
else:
    print(f"\n RESULT: Not statistically significant (p >= 0.05)")


***INTERPRETATION OF RESULTS:***
With p = 6.57e-25 (essentially zero), we have overwhelming evidence that
Group B's 14.07% conversion rate is significantly better than Group A's 5.4%.
The chi-square statistic of 106.23 is extremely high, confirming a massive
difference between groups. This is a clear winner - implement Group B!


In [ ]:
# 4 - Confidence Intervals
# -----------------------------------------------------------------------
# A single number like "14.07% conversion rate" doesn't tell the full story.
# Confidence intervals give us a range - we're 95% confident the true
# conversion rate falls somewhere between these two values.
#
# We're using the Wilson method here, which is more accurate than the
# basic method, especially when conversion rates are low or sample sizes vary.
#
# We also calculate the CI for the difference between the two groups.
# If that range doesn't include 0, we're confident B is genuinely better.
# The wider the range, the less precise our estimate (usually due to small sample size).

print("\n\n[2] CONFIDENCE INTERVALS (95%)")
print("-" * 80)

# Calculate conversion rates
conv_rate_a = conversions_a / len(group_a)
conv_rate_b = conversions_b / len(group_b)

# Calculate 95% confidence intervals using Wilson method
ci_a = proportion_confint(conversions_a, len(group_a), alpha=0.05, method='wilson')
ci_b = proportion_confint(conversions_b, len(group_b), alpha=0.05, method='wilson')

print(f"Group A Conversion Rate: {conv_rate_a*100:.2f}%")
print(f"  95% CI: [{ci_a[0]*100:.2f}%, {ci_a[1]*100:.2f}%]")
print(f"\nGroup B Conversion Rate: {conv_rate_b*100:.2f}%")
print(f"  95% CI: [{ci_b[0]*100:.2f}%, {ci_b[1]*100:.2f}%]")

# Calculate confidence interval for the difference between groups
diff = conv_rate_b - conv_rate_a
se_diff = np.sqrt((conv_rate_a * (1 - conv_rate_a) / len(group_a)) +
                   (conv_rate_b * (1 - conv_rate_b) / len(group_b)))
ci_diff_lower = diff - 1.96 * se_diff
ci_diff_upper = diff + 1.96 * se_diff

print(f"\nDifference (B - A): {diff*100:.2f} percentage points")
print(f"  95% CI: [{ci_diff_lower*100:.2f}pp, {ci_diff_upper*100:.2f}pp]")
print(f"\nRelative Lift: {(conv_rate_b/conv_rate_a - 1)*100:.1f}%")



***INTERPRETATION OF RESULTS:***
Group B's conversion rate (12.75% - 15.49%) is completely above Group A's
range (4.58% - 6.35%) with NO overlap - this confirms a clear winner.

The difference CI [7.04pp, 10.30pp] doesn't include 0, meaning we're 95%
confident Group B performs 7-10 percentage points better than Group A.

Relative lift of 160.5% means Group B converts 2.6x more users than Group A.
This is a massive improvement - definitely implement Group B's changes!


In [ ]:
# Cell 5 - Effect Size (Cohen's h)
# -----------------------------------------------------------------------
# The p-value only tells us IF the difference is real.
# Cohen's h tells us HOW BIG the difference actually is in practical terms.
# This matters because with enough data, even tiny differences become
# statistically significant - but that doesn't mean they're meaningful.
#
# Cohen's h is specifically designed for comparing two proportions/rates.
# It uses arcsin transformation to handle the fact that percentages
# behave differently than regular numbers, especially near 0% or 100%.
#
# Standard benchmarks: small = 0.2, medium = 0.5, large = 0.8
# These help us judge if the effect is practically important, not just statistically significant.

print("\n\n[3] EFFECT SIZE (COHEN'S H)")
print("-" * 80)

# Apply arcsin transformation to both proportions and calculate difference
p1 = conv_rate_a
p2 = conv_rate_b
cohens_h = 2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1)))

print(f"Cohen's h: {cohens_h:.4f}")
print(f"\nInterpretation:")
if abs(cohens_h) < 0.2:
    print("  Small effect size")
elif abs(cohens_h) < 0.5:
    print("  Medium effect size")
else:
    print("  Large effect size")

print(f"\nStandard benchmarks:")
print(f"  Small: h = 0.2")
print(f"  Medium: h = 0.5")
print(f"  Large: h = 0.8")


***INTERPRETATION OF RESULTS:***
Cohen's h = 0.2999 indicates a medium effect size, approaching the threshold
for "large" (0.5). Combined with our highly significant p-value, this tells us:
1. The difference IS real (from Chi-Square test)
2. The difference IS meaningful (from Cohen's h)

This is the sweet spot: statistically significant AND practically important.
Group B's improvement isn't just detectable - it's substantial enough to matter.


In [ ]:
# 6 T-Tests for Secondary Metrics
# -----------------------------------------------------------------------
# We already know Group B converts better, but we need to verify this isn't
# because Group B users just happened to browse more pages or spend more time.
# That would suggest a biased sample rather than a true treatment effect.
#
# We use independent t-tests here (not chi-square) because page views and
# time spent are continuous numerical variables, not binary yes/no outcomes.
#
# WHAT WE WANT TO SEE:
# If both groups show similar page views and time spent (p >= 0.05), it confirms
# the conversion lift is coming from the treatment itself, not pre-existing
# differences in user engagement. This validates our A/B test design.

print("\n[5] T-TESTS FOR SECONDARY METRICS")
print("-" * 80)

# Test for Page Views difference
page_views_a = group_a['Page Views']
page_views_b = group_b['Page Views']
t_stat_pv, p_value_pv = ttest_ind(page_views_a, page_views_b)

print(f"\nPage Views:")
print(f"  Group A mean: {page_views_a.mean():.2f}")
print(f"  Group B mean: {page_views_b.mean():.2f}")
print(f"  Difference: {page_views_b.mean() - page_views_a.mean():.2f}")
print(f"  T-statistic: {t_stat_pv:.4f}")
print(f"  P-value: {p_value_pv:.4f}")
if p_value_pv < 0.05:
    print(f" Statistically significant difference")
else:
    print(f" No significant difference (p >= 0.05)")

# Test for Time Spent difference
time_spent_a = group_a['Time Spent']
time_spent_b = group_b['Time Spent']
t_stat_ts, p_value_ts = ttest_ind(time_spent_a, time_spent_b)

print(f"\nTime Spent:")
print(f"  Group A mean: {time_spent_a.mean():.2f} seconds")
print(f"  Group B mean: {time_spent_b.mean():.2f} seconds")
print(f"  Difference: {time_spent_b.mean() - time_spent_a.mean():.2f} seconds")
print(f"  T-statistic: {t_stat_ts:.4f}")
print(f"  P-value: {p_value_ts:.4f}")
if p_value_ts < 0.05:
    print(f" Statistically significant difference")
else:
    print(f" No significant difference (p >= 0.05)")

print(f"\n💡 INSIGHT: Secondary metrics show minimal/no significant difference,")
print(f"   confirming conversion lift is NOT driven by increased engagement.")


***INTERPRETATION OF RESULTS:***
Page Views: p = 0.4360 (no significant difference)
Time Spent: p = 0.6387 (no significant difference)

Both groups had nearly identical engagement levels.
This confirms the 160% conversion lift is due to the treatment itself,
not because Group B users were more engaged to begin with.

This validates our A/B test - the groups were properly randomized and
the conversion improvement is a genuine effect of the changes in Group B.


In [ ]:
# ============================================================================
# EXECUTIVE SUMMARY
# ============================================================================

print("\n\n")
print("="*80)
print("EXECUTIVE SUMMARY")
print("="*80)

print(f"""
   CONVERSION RATES:
   Group A: {conv_rate_a*100:.2f}% (95% CI: {ci_a[0]*100:.2f}%-{ci_a[1]*100:.2f}%)
   Group B: {conv_rate_b*100:.2f}% (95% CI: {ci_b[0]*100:.2f}%-{ci_b[1]*100:.2f}%)
   
   Difference: +{diff*100:.2f} percentage points
   Relative Improvement: +{(conv_rate_b/conv_rate_a - 1)*100:.1f}%

   STATISTICAL SIGNIFICANCE:
   Chi-square test: χ² = {chi2:.2f}, p-value = {p_value:.2e}
   Result: Highly significant (p < 0.001)
   
   This means the difference is NOT due to random chance.

   EFFECT SIZE:
   Cohen's h = {cohens_h:.4f} (Medium effect)
   
   This shows the improvement is not just statistically significant,
   but also practically meaningful for the business.

   SECONDARY METRICS CHECK:
   Page Views: Group A = {page_views_a.mean():.2f}, Group B = {page_views_b.mean():.2f}
               → No significant difference (p = {p_value_pv:.3f})
   
   Time Spent: Group A = {time_spent_a.mean():.2f}s, Group B = {time_spent_b.mean():.2f}s
               → No significant difference (p = {p_value_ts:.3f})
   
    Key Insight: Both groups had similar engagement levels, which means
      the conversion improvement came from the treatment itself, not because
      users were browsing more or spending more time.

   FINAL RECOMMENDATION:

   IMPLEMENT GROUP B
   
   Reasons:
   • Group B increases conversions by {(conv_rate_b/conv_rate_a - 1)*100:.0f}% compared to Group A
   • The result is statistically significant (p < 0.001)
   • The improvement is consistent and reliable
   • No negative impact on user engagement metrics
   
   Expected Impact:
   If we roll out Group B to all users, we can expect conversion rates
   to improve from {conv_rate_a*100:.1f}% to approximately {conv_rate_b*100:.1f}%.
""")
